# 03 · Fixed LGP vs the GDD thermal clock

**Question.** The pipeline gives maize a fixed calendar season from `config/crop_coefficients.yaml` (120 d main, 90 d short), identical everywhere. `src/gdd_clock.py` already derives a per-pixel thermal season from ERA5-Land GDD anchored on SOS, with maturity targets seeded per pixel from the AEZ class. How far apart are they, and does it matter?

**How to run:** put the `planting_pipeline` folder on your Google Drive, run top-to-bottom, approve the Drive-mount and Earth-Engine prompts. Export cells start GEE tasks and return immediately; the scoring cells read the CSVs once the tasks finish (watch https://code.earthengine.google.com/tasks).

## Setup

### Stage 0 · Runtime

Installs the Earth Engine client, geemap, pandas, geopandas and scipy. `scipy` is the one that matters
here: every score in this notebook is a leave-one-out cross-validation with a paired bootstrap interval,
and both come from scipy.

**Expected output.** `installed.`

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas scipy 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine

`EE ready: ok`. The export cells below submit **batch tasks** and return immediately; the scoring cells
read the resulting CSVs. Between the two you have to wait, and you can close the browser while you do.
Watch the queue at code.earthengine.google.com/tasks.

**The export queue is per cloud project.** `ee-manzikye` has left batches in READY for hours. If the
tasks are not entering RUNNING within about 20 minutes, switch `PROJECT` to
`indigo-proxy-484220-q8` and resubmit rather than waiting.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Drive

`pipeline on path: ...`. The scoring scripts read and write `Cropyield-Data/` inside this folder, so the
notebook must `chdir` here for the relative paths to resolve.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Scoring convention used throughout
`n` is small (6–81 zones) in every test here, and a single 70/30 split at n≈45 has a **±0.10 t/ha standard deviation — larger than any effect measured**. So every test below uses **leave-one-out CV** (each free parameter refit on n−1) plus a **paired bootstrap** CI, and reports **Spearman** alongside MAE because rank skill is invariant to the yield ceiling Ym. Reporting a single split would have produced two false positives in this round.

## Export
Exports season length **and flowering timing** — a shifted flowering dekad moves the critical-window water balance even when total length agrees, and CPI puts the FAO-33 Ky=1.5 weight on flowering.

> **Two compute traps hit here.** (a) The short-rains variant timed out: the production short-rains planting gap-fills with a **44-year CHIRPS climatology**, which on top of 27 dekads of ERA5-Land exceeds the EE budget — fixed with a light onset and `dk_hi = se+14`. (b) `gdd_maturity_from_aez` builds a very large client-side graph and stalled submission; short rains now use the constant **early-class 1300 °C·d**, which is the right target for short-duration maize anyway.

### Stage 1 · Export season length and flowering timing

**What runs.** Three variants. Each exports two things, and the second is the one people forget:
**season length** and **flowering timing**. A shifted flowering dekad moves the critical-window water
balance even when total length agrees, and CPI puts the FAO-33 weight of $K_y = 1.5$ on flowering.

**Two compute traps hit here, both now fixed in the scripts.**

1. The short-rains variant timed out. The production short-rains planting gap-fills with a **44-year
   CHIRPS climatology**, which on top of 27 dekads of ERA5-Land exceeds the Earth Engine budget. Fixed
   with a light onset and `dk_hi = se + 14`.
2. `gdd_maturity_from_aez` builds a very large client-side graph and stalled submission. Short rains now
   use the constant early-class target of **1300 °C·d**, which is the right target for short-duration
   maize anyway.

**Expected output.** Task submission lines. Allow time before scoring.

In [ ]:
!python lgp_vs_gdd.py --variant ke_long
!python lgp_vs_gdd.py --variant ke_short
!python lgp_vs_gdd.py --variant et_meher

### Stage 2 · Score, and the largest discrepancy in the whole pipeline

**What runs.** Compares the fixed 120-day season against the per-pixel thermal season from ERA5-Land
growing degree days, anchored on the detected start of season.

**Expected values, Kenya long rains 2024.** The GDD county mean is 137 days, but **pixel-weighted over
actual maize it is 173 days**. Only **8 of 44** counties fall within ±15 days of the fixed assumption.

| | GDD season | vs fixed 120 d | Flowering shift |
|---|---|---|---|
| Highland, 1800 m and above, n = 14 | **187.7 d** | **+67.7 d** | **+25.9 d later** |
| Lowland, below 1800 m, n = 30 | 113.7 d | −6.3 d | −12.7 d earlier |

The overruns are the grain basket: Nyandarua +112, Bomet +100, Kericho +84, Uasin Gishu +79. The
shortfalls are the hot arid and semi-arid lands: Turkana −55, Marsabit −50, Tana River −46.
Ethiopia Meher: GDD 142 days, 154 pixel-weighted, +22 days, with 3 of 6 regions within ±15 days.

**Why this matters more than the other tests.** The fixed cycle is wrong by more than two months in the
highlands, and the flowering window moves by nearly a month with it. Because the stage weights put
three times the weight on flowering, the water balance is being read at the wrong point of the season
exactly where most of Kenya's maize is grown.

**And why the fix is not simply to adopt the GDD clock.** Notebook 02 shows a zone-aware **season
length** made the yield worse, and the CHIRTS work shows the spreadsheet GDD targets are worse than the
ERA5 ones. The evidence supports replacing the **fixed 120-day cycle**, not adopting the current
maturity targets uncritically. Treat this as the strongest open item in the methodology, not as a
settled change.

In [ ]:
!python lgp_vs_gdd_score.py

## Result — Kenya Long rains 2024

Fixed 120 d. GDD county mean 137 d, but **pixel-weighted over actual maize 173 d**. Only **8 of 44** counties within ±15 d.

| | GDD LGP | vs fixed | flowering shift |
|---|---|---|---|
| highland ≥1800 m (n=14) | **187.7 d** | **+67.7 d** | **+25.9 d later** |
| lowland <1800 m (n=30) | 113.7 d | −6.3 d | −12.7 d earlier |

Overruns are the grain basket (Nyandarua +112, Bomet +100, Kericho +84, Uasin Gishu +79); shortfalls are hot ASAL (Turkana −55, Marsabit −50, Tana River −46).

**Ethiopia Meher:** GDD 142 d (pixel-wtd 154), +22 d; 3/6 regions within ±15 d. SNNP +47, Oromia +41, Amhara +39; Gambela +0, Benshangul −2.

### Why this is the most consequential finding of the round
1. The fixed LGP errs in **opposite directions** highland vs lowland — no single national value can fix it.
2. Highland **flowering lands 26 d later** than the Kc curve assumes, so the Ky=1.5 critical-window weight is currently applied to roughly the wrong three dekads there. That is plausibly a larger error than season length itself.
3. **It explains the failed zone-aware LGP A/B** (notebook 02). That test gave the highland 180 d — close to the GDD answer of 188 d — and still lost, because it kept the Mar–May MAM rainfall window: 180 d on MAM just accumulates deficit past the rains. The correct experiment is a highland **unimodal Mar–Aug window at ~188 d**, exactly what `lgp_ab_test_MAM.md` hypothesised but could not test.